<a href="https://colab.research.google.com/github/waxsd100/meikey/blob/master/colab/meikey.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# meikey（Google Colab）

公開鍵に好きな文字列が入った ed25519 の SSH 鍵を、Colab の GPU と CPU で探します。上のセルから順に実行してください。

- GPU は **ランタイム → ランタイムのタイプを変更** で選びます（速さは [ベンチマーク](https://github.com/waxsd100/meikey/blob/master/docs/benchmark.md)）
- 使い終わったら **ランタイム → ランタイムを接続解除して削除** してください
- 秘密鍵はパスフレーズなしで作られます。ダウンロードしたら手順 8 で VM から消し、手元で `ssh-keygen -p -f <秘密鍵>` を実行してパスフレーズを付けてください

## 1. GPU と CPU を確認する

In [ ]:
#@title GPU・CPU・メモリなどを表示する
import os, platform, shutil, subprocess, unicodedata

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=10)
    return r.stdout.strip() if r.returncode == 0 else ""

def row(label, value):
    # meikey の表示と同じく、ラベルを表示幅 14 桁にそろえる
    width = sum(2 if unicodedata.east_asian_width(c) in "WF" else 1 for c in label)
    print(label + " " * max(14 - width, 1) + value)

def known(v):
    return v and v not in ("[N/A]", "N/A", "[Not Supported]")

print("---- GPU")
fields = ["name", "compute_cap", "memory.total", "memory.used", "driver_version", "power.limit",
          "clocks.max.sm", "clocks.max.mem", "pcie.link.gen.max", "pcie.link.width.max", "temperature.gpu"]
out = sh(f"nvidia-smi --query-gpu={','.join(fields)} --format=csv,noheader,nounits")
if out:
    cuda = sh("nvidia-smi | sed -n 's/.*CUDA Version: *\\([0-9.]*\\).*/\\1/p'")
    gpus = out.splitlines()
    for i, line in enumerate(gpus):
        g = dict(zip(fields, (x.strip() for x in line.split(","))))
        row("GPU" if len(gpus) == 1 else f"GPU {i}", g["name"])
        if known(g["compute_cap"]):
            row("世代", "sm_" + g["compute_cap"].replace(".", ""))
        row("メモリ", f"{int(g['memory.total']) / 1024:.1f} GiB（使用中 {g['memory.used']} MiB）")
        row("ドライバ", g["driver_version"] + (f"（CUDA {cuda}）" if cuda else ""))
        if known(g["power.limit"]):
            row("電力上限", f"{float(g['power.limit']):.0f} W")
        if known(g["clocks.max.sm"]):
            row("最大クロック", f"SM {g['clocks.max.sm']} MHz・メモリ {g['clocks.max.mem']} MHz")
        if known(g["pcie.link.gen.max"]):
            row("PCIe", f"Gen{g['pcie.link.gen.max']} x{g['pcie.link.width.max']}")
        if known(g["temperature.gpu"]):
            row("温度", f"{g['temperature.gpu']} ℃")
else:
    row("GPU", "なし（ランタイムのタイプで GPU を選ぶ）")

print("---- CPU")
lscpu = dict(l.split(":", 1) for l in sh("LC_ALL=C lscpu").splitlines() if ":" in l)
lscpu = {k.strip(): v.strip() for k, v in lscpu.items()}
flags = set(sh("grep -m1 '^flags' /proc/cpuinfo").split())
row("CPU", lscpu.get("Model name", "?"))
sockets = int(lscpu.get("Socket(s)", "1") or 1)
cores = int(lscpu.get("Core(s) per socket", "0") or 0) * sockets
row("コア", f"{cores} コア・{os.cpu_count()} スレッド（ソケット {sockets}）")
mhz = lscpu.get("CPU max MHz") or sh("grep -m1 'cpu MHz' /proc/cpuinfo | cut -d: -f2")
if mhz:
    row("クロック", f"{float(mhz):.0f} MHz")
caches = [f"{k.split()[0]} {lscpu[k].split(' (')[0]}" for k in ("L2 cache", "L3 cache") if k in lscpu]
if caches:
    row("キャッシュ", "・".join(caches))
marks = [("AVX2", "avx2"), ("AVX-512", "avx512f"), ("AVX-512 IFMA", "avx512ifma"), ("AMX", "amx_tile")]
row("命令", "  ".join(f"{name} {'✓' if flag in flags else '✗'}" for name, flag in marks))
if "Hypervisor vendor" in lscpu:
    row("仮想化", lscpu["Hypervisor vendor"])

print("---- その他")
machine = sh("curl -s -m 1 -H 'Metadata-Flavor: Google' "
             "http://metadata.google.internal/computeMetadata/v1/instance/machine-type")
if machine and "/" in machine:
    row("マシン", machine.rsplit("/", 1)[-1])
mem = {l.split(":")[0]: int(l.split()[1]) for l in open("/proc/meminfo") if l.split()[1].isdigit()}
row("メモリ", f"{mem['MemTotal'] / 2**20:.1f} GiB（空き {mem['MemAvailable'] / 2**20:.1f} GiB）")
disk = shutil.disk_usage("/content" if os.path.exists("/content") else "/")
row("ディスク", f"{disk.total / 2**30:.0f} GiB（空き {disk.free / 2**30:.0f} GiB）")
try:
    os_name = platform.freedesktop_os_release().get("PRETTY_NAME", "")
except OSError:
    os_name = platform.system()
row("OS", f"{os_name}（カーネル {platform.release()}）")
row("Python", platform.python_version())

## 2. Rust を入れる

In [ ]:
import os, shutil

CARGO_BIN = os.path.expanduser("~/.cargo/bin")
if not os.path.exists(os.path.join(CARGO_BIN, "cargo")):
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal --default-toolchain none
if CARGO_BIN not in os.environ["PATH"]:
    os.environ["PATH"] = CARGO_BIN + ":" + os.environ["PATH"]
!rustup --version

## 3. meikey をビルドする

初回は数分かかります。

In [ ]:
REPO = "https://github.com/waxsd100/meikey"
BRANCH = "master"

if os.path.exists("/content/meikey"):
    !git -C /content/meikey pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO} /content/meikey
%cd /content/meikey
# rust-toolchain.toml に書いたバージョン（1.99）を入れる
!rustup toolchain install
!RUSTFLAGS="-C target-cpu=native" cargo build --release
BIN = "/content/meikey/target/release/meikey"

## 4. GPU と CPU が使えるか確かめる

In [ ]:
import glob

# libnvrtc があれば使えるようにする（ドライバが古くて、埋め込んだ PTX を読めないとき用）
candidates = sorted(
    p for d in ["/usr/local/cuda/lib64", "/usr/local/cuda/targets/x86_64-linux/lib", "/usr/lib/x86_64-linux-gnu"]
              + glob.glob("/usr/local/lib/python3*/dist-packages/nvidia/cuda_nvrtc/lib")
              + glob.glob("/usr/local/lib/python3*/dist-packages/nvidia/cu*/lib")
    for p in glob.glob(d + "/libnvrtc.so*")
    if "builtins" not in p and ".alt." not in p
)
LIBDIR = "/content/meikey-libs"
os.makedirs(LIBDIR, exist_ok=True)
if candidates:
    nvrtc = candidates[-1]
    link = os.path.join(LIBDIR, "libnvrtc.so")
    if os.path.lexists(link):
        os.remove(link)
    os.symlink(nvrtc, link)
    # libnvrtc が読み込む libnvrtc-builtins も同じディレクトリから探せるように
    paths = [LIBDIR, os.path.dirname(nvrtc)]
    old = os.environ.get("LD_LIBRARY_PATH", "")
    os.environ["LD_LIBRARY_PATH"] = ":".join(paths + [p for p in old.split(":") if p and p not in paths])


import subprocess, codecs

def meikey(*args):
    # meikey を実行して、出力をそのまま（進捗の行の上書きも含めて）表示する
    p = subprocess.Popen([BIN, *map(str, args)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    dec = codecs.getincrementaldecoder("utf-8")("replace")
    while chunk := os.read(p.stdout.fileno(), 4096):
        print(dec.decode(chunk), end="", flush=True)
    return p.wait()

# 探すのに使うもの（手順 5・6 のフォームの選択肢）と、meikey に付ける引数
DEVICES = {"GPU": ["--low-security", "--gpu"], "CPU": [], "GPU + CPU": ["--low-security", "--both"]}

if meikey("--check", "--low-security", "--both") != 0:
    print("GPU は使えない。CPU だけで探せる:")
    meikey("--check")

### （任意）GPU の計算のテスト

テスト用のビルドに数分かかります。

In [ ]:
!MEIKEY_REQUIRE_SIMD=cuda cargo test --release --lib service::gpu

## 5. 速さを測る

「全部」なら GPU・CPU・GPU + CPU を順に測ります。

In [ ]:
TARGET = "wakokara"  #@param {type:"string"}
SECONDS = 10  #@param {type:"integer"}
DEVICE = "全部"  #@param ["全部", "GPU", "CPU", "GPU + CPU"]

for name in (DEVICES if DEVICE == "全部" else [DEVICE]):
    print(f"---- {name}")
    meikey(*DEVICES[name], "--bench", SECONDS, TARGET)

## 6. 鍵を探す

見つかると、`/content/keys/` に秘密鍵と公開鍵（`.pub`）を書き出します。

In [ ]:
TARGET = "wakokara"  #@param {type:"string"}
CASE_SENSITIVE = False  #@param {type:"boolean"}
COMMENT = ""  #@param {type:"string"}
DEVICE = "GPU + CPU"  #@param ["GPU", "CPU", "GPU + CPU"]

os.makedirs("/content/keys", exist_ok=True)
OUTPUT = f"/content/keys/id_ed25519_{TARGET}"
args = [*DEVICES[DEVICE], TARGET, "-o", OUTPUT, "-C", COMMENT]
if CASE_SENSITIVE:
    args.append("--case-sensitive")
meikey(*args)

## 7. 鍵をダウンロードする

In [ ]:
from google.colab import files

files.download(OUTPUT)
files.download(OUTPUT + ".pub")

## 8. VM から鍵を消す

ダウンロードできたのを確かめてから実行してください。

In [ ]:
!shred -u {OUTPUT} && rm -f {OUTPUT}.pub && ls -la /content/keys